## 4. Inference

In [17]:
!pip install git+https://github.com/psychemedia/ipython_magic_folium.git

  Cloning https://github.com/psychemedia/ipython_magic_folium.git to /tmp/pip-req-build-pE2LWZ
  Running setup.py bdist_wheel for ipython-magic-folium ... done
  Stored in directory: /tmp/pip-ephem-wheel-cache-xFnIJk/wheels/7c/49/f8/8577b6d1b8fcaa766dd7c6f7996f22a913e4a260443e75a095
Successfully built ipython-magic-folium
You are using pip version 10.0.1, however version 19.0.2 is available.
You should consider upgrading via the 'pip install --upgrade pip' command.


In [7]:
import sagemaker
import numpy as np
import pandas as pd

from sagemaker.amazon.common import numpy_to_record_serializer, record_deserializer
from sagemaker.predictor import RealTimePredictor

sess = sagemaker.Session()

In [13]:
ENDPOINT = "crime-prediction-bogota-summit"

cluster = pd.read_csv("cluster_location-co.csv")
crime = pd.read_csv('crime-colombia.csv', header=0)
xref = pd.DataFrame(crime['City'])

###### read data dynamically
#slice=crimeArray[:,1:5]
#slice.shape
#slice
#s=slice[2:3]

##### read data statically
s = np.array([[4, 8.1, 294, 80]], np.float32)

predictor = RealTimePredictor(endpoint=ENDPOINT, 
                              sagemaker_session=sess, serializer=numpy_to_record_serializer(),
                              deserializer=record_deserializer())

result = predictor.predict(s)

In [14]:
clusters = [r.label['closest_cluster'].float32_tensor.values[0] for r in result]
i = 0
    
for r in result:
    out = {
      "City" : xref['City'].iloc[i],
      "closest_cluster" :  r.label['closest_cluster'].float32_tensor.values[0],
      "crimeCluster" : crime['crimeCluster'].iloc[i],  
      "Kidnapping" :  crime['Kidnapping'].iloc[i],
      "Latitude"   :  cluster['lat'][r.label['closest_cluster'].float32_tensor.values[0]],
      "Longitude"  :  cluster['long'][r.label['closest_cluster'].float32_tensor.values[0]]
    }    
    print(out) 
    i = i + 1

{'City': 'Bogota', 'Kidnapping': 21.2, 'crimeCluster': 4, 'Longitude': -74.07380461692809, 'closest_cluster': 6.0, 'Latitude': 4.598915389484102}


In [15]:
%load_ext folium_magic

# Set marker
marker = [[4.600851053970031,-74.07008171081543,'Operacion de equipo mas cercano!'],[
    cluster['lat'][r.label['closest_cluster'].float32_tensor.values[0]],
    cluster['long'][r.label['closest_cluster'].float32_tensor.values[0]],'Proxima area para un crimen!']]

#marker = [[4.600851053970031,-74.07008171081543,'Operacion de equipo mas cercano!'],[
#           4.598915389484102,-74.0738046169281,'Proxima area para un crimen!']]

%folium_map -l 4.60089383102632,-74.07192707061768 -z 17 -M marker -g location-bogota.json

The folium_magic extension is already loaded. To reload it, use:
  %reload_ext folium_magic
